# HW2 — New Models & Ensemble: Irrigation Need Prediction (S6E4)
**Tyler Wolf Williams**

**Run `HW2_FeatureEngineering_Tyler.ipynb` first** — this notebook loads the engineered feature set it produces.

## Models in this notebook
| # | Model | Why it's different from Phase 1 & 2 |
|---|-------|--------------------------------------|
| 1 | **MLP Neural Network** | Gradient descent, not greedy tree splits; learns shared hidden representations; needs feature scaling |
| 2 | **HistGradientBoostingClassifier** | sklearn's independent GBM implementation; histogram-based splitting; native class weighting; different regularization |
| 3 | **LightGBM DART** | Applies neural-network-style dropout to boosting; drops random previous trees per iteration; prevents tree dominance |
| 4 | **Stacking ensemble** | Trains LR meta-learner on out-of-fold probability predictions from the 3 models above |
| 5 | **Probability averaging** | Equal-weight and custom-weight average of all model probabilities |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from scipy.stats import uniform as sp_uniform
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)

# Load engineered data produced by the feature engineering notebook
train = pd.read_csv('../Homework 2/train_engineered.csv')
test  = pd.read_csv('../Homework 2/test_engineered.csv')

with open('../Homework 2/selected_features.txt') as fh:
    feature_cols = fh.read().splitlines()

TARGET        = 'target'
ID_COL        = 'id'
inv_label_map = {0: 'Low', 1: 'Medium', 2: 'High'}

X      = train[feature_cols].values
y      = train[TARGET].values
X_test = test[feature_cols].values

print(f'Features: {len(feature_cols)}')
print(f'Train:    {X.shape[0]:,}    Test: {X_test.shape[0]:,}')
print('\nClass distribution:')
for cls, name in inv_label_map.items():
    pct = (y == cls).mean() * 100
    print(f'  {name}: {(y == cls).sum():,} ({pct:.1f}%)')

In [ ]:
# ─── Shared helpers ───────────────────────────────────────────────────────────

def evaluate_cv(model, X_data, y_data, label, cv=5):
    """Stratified K-fold CV — returns mean accuracy and F1 macro."""
    kf  = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    acc = cross_val_score(model, X_data, y_data, cv=kf, scoring='accuracy', n_jobs=-1)
    f1  = cross_val_score(model, X_data, y_data, cv=kf, scoring='f1_macro',  n_jobs=-1)
    print(f'{label}')
    print(f'  CV Accuracy : {acc.mean():.4f} ± {acc.std():.4f}')
    print(f'  CV F1 Macro : {f1.mean():.4f} ± {f1.std():.4f}')
    return acc.mean(), f1.mean()

def holdout_eval(model, X_data, y_data, label, test_size=0.2):
    """Single stratified holdout — faster alternative to full CV."""
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    tr_idx, val_idx = next(sss.split(X_data, y_data))
    model.fit(X_data[tr_idx], y_data[tr_idx])
    preds = model.predict(X_data[val_idx])
    acc   = accuracy_score(y_data[val_idx], preds)
    f1    = f1_score(y_data[val_idx], preds, average='macro')
    print(f'{label}')
    print(f'  Holdout Accuracy : {acc:.4f}')
    print(f'  Holdout F1 Macro : {f1:.4f}')
    return acc, f1

def make_submission(ids, preds, filename):
    sub = pd.DataFrame({'id': ids, 'Irrigation_Need': [inv_label_map[p] for p in preds]})
    sub.to_csv(f'../Homework 2/{filename}', index=False)
    dist = sub['Irrigation_Need'].value_counts().to_dict()
    print(f'  Saved {filename} | {dist}')
    return sub

print('Helpers ready.')

---
## Model 1: MLP Neural Network

**Why MLP?**
- Learns via gradient descent through continuous weight updates — fundamentally different from greedy tree splits
- Hidden layers learn *shared representations* across classes (unlike one-vs-rest trees)
- Better at smooth, dense decision boundaries; tree models excel at axis-aligned splits
- Sensitive to feature scale → requires StandardScaler
- Class imbalance handled via `class_weight`

**Note:** Full 5-fold CV on 630K rows is very slow for neural networks (~30 min/config). We tune on a stratified 150K subset, then evaluate the winner on full data with 3-fold CV.

In [ ]:
print('=' * 60)
print('MODEL 1: MLP Neural Network')
print('=' * 60)

from sklearn.model_selection import train_test_split

# Scale features (required for MLP gradient descent)
scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# 50K stratified sample — fast enough for 3-config comparison (~2 min total)
X_sub, _, y_sub, _ = train_test_split(
    X_scaled, y, train_size=50_000, stratify=y, random_state=42
)
X_sub_tr, X_sub_val, y_sub_tr, y_sub_val = train_test_split(
    X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42
)
print(f'Tuning subset: {len(X_sub_tr):,} train / {len(X_sub_val):,} val\n')

# 3 fixed configs — single holdout comparison
configs = [
    {'hidden_layer_sizes': (128, 64),      'alpha': 0.001, 'learning_rate_init': 0.005},
    {'hidden_layer_sizes': (256, 128),     'alpha': 0.001, 'learning_rate_init': 0.001},
    {'hidden_layer_sizes': (256, 128, 64), 'alpha': 0.01,  'learning_rate_init': 0.005},
]

best_mlp_f1     = -1
best_mlp_params = None
for cfg in configs:
    mlp = MLPClassifier(
        **cfg, activation='relu', solver='adam', batch_size=4096,
        max_iter=40, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=8, random_state=42
    )
    mlp.fit(X_sub_tr, y_sub_tr)
    preds = mlp.predict(X_sub_val)
    f1    = f1_score(y_sub_val, preds, average='macro')
    acc   = accuracy_score(y_sub_val, preds)
    print(f"  layers={str(cfg['hidden_layer_sizes']):<18} alpha={cfg['alpha']}  "
          f"lr={cfg['learning_rate_init']}  Acc={acc:.4f}  F1={f1:.4f}")
    if f1 > best_mlp_f1:
        best_mlp_f1     = f1
        best_mlp_params = cfg

print(f'\nBest config: {best_mlp_params}')

In [ ]:
# Train best config on 80% for holdout report, then 100% for Kaggle submission (~2 min)
X_tr_m, X_val_m, y_tr_m, y_val_m = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

mlp_tuned = MLPClassifier(
    **best_mlp_params, activation='relu', solver='adam', batch_size=4096,
    max_iter=60, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=10, random_state=42
)
mlp_tuned.fit(X_tr_m, y_tr_m)
mlp_val_preds      = mlp_tuned.predict(X_val_m)
mlp_acc            = accuracy_score(y_val_m, mlp_val_preds)
mlp_f1             = f1_score(y_val_m, mlp_val_preds, average='macro')
mlp_base_acc, mlp_base_f1 = mlp_acc, mlp_f1   # no separate baseline run

print(f'MLP Tuned — Holdout Acc={mlp_acc:.4f}  F1={mlp_f1:.4f}')
print(classification_report(y_val_m, mlp_val_preds, target_names=['Low', 'Medium', 'High']))

# Retrain on full data for Kaggle submission
mlp_final = MLPClassifier(
    **best_mlp_params, activation='relu', solver='adam', batch_size=4096,
    max_iter=60, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=10, random_state=42
)
mlp_final.fit(X_scaled, y)
mlp_preds = mlp_final.predict(X_test_scaled)
mlp_probs = mlp_final.predict_proba(X_test_scaled)
print('\nMLP submission:')
make_submission(test[ID_COL], mlp_preds, 'submission_mlp_hw2.csv')

In [ ]:
print(f'MLP Summary | Architecture: {best_mlp_params["hidden_layer_sizes"]} '
      f'| alpha={best_mlp_params["alpha"]} | lr={best_mlp_params["learning_rate_init"]}')
print(f'            | Holdout Acc={mlp_acc:.4f}  F1={mlp_f1:.4f}')

---
## Model 2: HistGradientBoostingClassifier

**Why HistGBM?**
- sklearn's *own* gradient boosting implementation — completely independent code from LightGBM/XGBoost
- Uses a different histogram-binning strategy (fixed 255 bins by default vs. LGB's adaptive)
- Native `class_weight='balanced'` support in the GBM objective itself (not post-hoc)
- Supports monotonic constraints and interaction constraints
- Deterministic even with parallel training (unlike LGB with multiple threads)
- `l2_regularization` penalizes leaf weights in a different way than LGB's `reg_lambda`

In [ ]:
print('=' * 60)
print('MODEL 2: HistGradientBoostingClassifier')
print('=' * 60)

# Single holdout baseline — faster than 5-fold CV (~30 sec)
Xt_h, Xv_h, yt_h, yv_h = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

hgb_baseline = HistGradientBoostingClassifier(
    max_iter=100, learning_rate=0.1, max_depth=6,
    class_weight='balanced', random_state=42
)
hgb_baseline.fit(Xt_h, yt_h)
bp = hgb_baseline.predict(Xv_h)
hgb_base_acc = accuracy_score(yv_h, bp)
hgb_base_f1  = f1_score(yv_h, bp, average='macro')
print(f'\nHGB Baseline — Holdout Acc={hgb_base_acc:.4f}  F1={hgb_base_f1:.4f}')

In [ ]:
# 5 configs × 3-fold CV, max_iter=100 — ~2-3 min with n_jobs=-1
print('Tuning HGB (5 configs × 3-fold, max_iter=100)...\n')

hgb_param_dist = {
    'learning_rate':     [0.05, 0.1, 0.15],
    'max_depth':         [4, 6, 8],
    'min_samples_leaf':  [50, 100],
    'l2_regularization': [0.0, 0.5, 1.0],
}

hgb_search = RandomizedSearchCV(
    HistGradientBoostingClassifier(max_iter=100, class_weight='balanced', random_state=42),
    param_distributions=hgb_param_dist,
    n_iter=5, cv=3, scoring='f1_macro',
    random_state=42, n_jobs=-1, verbose=1
)
hgb_search.fit(X, y)
print(f'\nBest HGB params:  {hgb_search.best_params_}')
print(f'Best CV F1 Macro: {hgb_search.best_score_:.4f}')
hgb_best = hgb_search.best_estimator_

In [ ]:
# Holdout eval + submission (~1 min)
hgb_best_eval = HistGradientBoostingClassifier(**hgb_best.get_params())
hgb_best_eval.fit(Xt_h, yt_h)
hgb_val_preds = hgb_best_eval.predict(Xv_h)
hgb_acc = accuracy_score(yv_h, hgb_val_preds)
hgb_f1  = f1_score(yv_h, hgb_val_preds, average='macro')
print(f'HGB Tuned — Holdout Acc={hgb_acc:.4f}  F1={hgb_f1:.4f}')
print(classification_report(yv_h, hgb_val_preds, target_names=['Low', 'Medium', 'High']))

hgb_best.fit(X, y)
hgb_preds = hgb_best.predict(X_test)
hgb_probs = hgb_best.predict_proba(X_test)
print('\nHGB submission:')
make_submission(test[ID_COL], hgb_preds, 'submission_histgbm_hw2.csv')

---
## Model 3: LightGBM DART (Dropout Additive Regression Trees)

**Why DART?**
- Same LightGBM library but fundamentally different boosting algorithm
- Standard boosting (`gbdt`) is greedy: each tree fits residuals of *all* prior trees
- DART applies **dropout from neural networks**: at each iteration, randomly selected previous trees are *dropped* (excluded) before fitting the next tree
- This prevents early trees from dominating the ensemble and acts as strong regularization
- Especially beneficial when the model is likely to overfit on majority classes
- `drop_rate` and `skip_drop` are DART-specific hyperparameters not present in gbdt/goss

In [ ]:
print('=' * 60)
print('MODEL 3: LightGBM DART (Dropout Boosting)')
print('=' * 60)

# 3-fold CV for baseline — n_estimators=100 keeps each fold fast (~2 min)
kf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
dart_base_model = lgb.LGBMClassifier(
    boosting_type='dart', n_estimators=100, learning_rate=0.1, num_leaves=63,
    drop_rate=0.1, skip_drop=0.5,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)
acc3 = cross_val_score(dart_base_model, X, y, cv=kf3, scoring='accuracy', n_jobs=-1)
f1_3 = cross_val_score(dart_base_model, X, y, cv=kf3, scoring='f1_macro',  n_jobs=-1)
dart_base_acc, dart_base_f1 = acc3.mean(), f1_3.mean()
print(f'\nDART Baseline (3-fold) — Acc={dart_base_acc:.4f}±{acc3.std():.4f}  '
      f'F1={dart_base_f1:.4f}±{f1_3.std():.4f}')

In [ ]:
# 3 fixed configs on a single holdout — fast (~1-2 min total)
Xt_d, Xv_d, yt_d, yv_d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

dart_configs = [
    {'num_leaves': 63,  'drop_rate': 0.10, 'skip_drop': 0.50, 'reg_lambda': 0.0},
    {'num_leaves': 63,  'drop_rate': 0.20, 'skip_drop': 0.40, 'reg_lambda': 0.5},
    {'num_leaves': 95,  'drop_rate': 0.15, 'skip_drop': 0.50, 'reg_lambda': 1.0},
]

best_dart_f1  = -1
best_dart_cfg = None
for cfg in dart_configs:
    m = lgb.LGBMClassifier(
        boosting_type='dart', n_estimators=100, learning_rate=0.1,
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1, **cfg
    )
    m.fit(Xt_d, yt_d)
    preds = m.predict(Xv_d)
    f1    = f1_score(yv_d, preds, average='macro')
    acc   = accuracy_score(yv_d, preds)
    print(f"  leaves={cfg['num_leaves']}  drop={cfg['drop_rate']}  "
          f"skip={cfg['skip_drop']}  lambda={cfg['reg_lambda']}  Acc={acc:.4f}  F1={f1:.4f}")
    if f1 > best_dart_f1:
        best_dart_f1  = f1
        best_dart_cfg = cfg

print(f'\nBest DART config: {best_dart_cfg}  (F1={best_dart_f1:.4f})')

In [ ]:
# Holdout eval + full-data train + submission (~1-2 min)
dart_best = lgb.LGBMClassifier(
    boosting_type='dart', n_estimators=100, learning_rate=0.1,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
    **best_dart_cfg
)

# Eval on same holdout used for tuning
dart_best.fit(Xt_d, yt_d)
dart_val_preds = dart_best.predict(Xv_d)
dart_acc = accuracy_score(yv_d, dart_val_preds)
dart_f1  = f1_score(yv_d, dart_val_preds, average='macro')
print(f'DART Tuned — Holdout Acc={dart_acc:.4f}  F1={dart_f1:.4f}')
print(classification_report(yv_d, dart_val_preds, target_names=['Low', 'Medium', 'High']))

# Train on full data
dart_full = lgb.LGBMClassifier(
    boosting_type='dart', n_estimators=100, learning_rate=0.1,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
    **best_dart_cfg
)
dart_full.fit(X, y)
dart_preds = dart_full.predict(X_test)
dart_probs = dart_full.predict_proba(X_test)
print('\nDART submission:')
make_submission(test[ID_COL], dart_preds, 'submission_dart_hw2.csv')

---
## Stacking Ensemble (Meta-Learner)

**How stacking works:**
1. Each base model (HGB, DART, MLP) generates **out-of-fold probability predictions** using 3-fold CV
2. These OOF probabilities (3 models × 3 classes = 9 meta-features) are fed as inputs to a **Logistic Regression meta-learner**
3. The meta-learner learns the optimal *linear combination* of base model confidence scores
4. For test predictions: each base model is retrained on all training data, probabilities passed to meta-learner

**Why stacking > simple averaging:** The meta-learner can down-weight models that are overconfident or systematically wrong on certain classes (e.g., the rare 'High' class).

In [ ]:
print('=' * 60)
print('STACKING ENSEMBLE (HGB + DART, manual OOF)')
print('=' * 60)
print('3-fold OOF with fast configs — ~2 min\n')
# MLP is excluded from the stacking fold loop (too slow to retrain 3× on full data).
# It contributes via probability averaging in the next cell instead.

kf_s     = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
oof_hgb  = np.zeros((len(X), 3))
oof_dart = np.zeros((len(X), 3))
tst_hgb  = np.zeros((len(X_test), 3))
tst_dart = np.zeros((len(X_test), 3))

hgb_s  = HistGradientBoostingClassifier(
    max_iter=75, learning_rate=0.1, max_depth=6,
    class_weight='balanced', random_state=42
)
dart_s = lgb.LGBMClassifier(
    boosting_type='dart', n_estimators=75, learning_rate=0.1, num_leaves=63,
    drop_rate=0.15, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
)

for fold, (tr_idx, val_idx) in enumerate(kf_s.split(X, y)):
    print(f'  Fold {fold+1}/3...')
    Xf_tr, Xf_val = X[tr_idx], X[val_idx]
    yf_tr, yf_val = y[tr_idx], y[val_idx]

    hgb_s.fit(Xf_tr, yf_tr)
    oof_hgb[val_idx] = hgb_s.predict_proba(Xf_val)
    tst_hgb         += hgb_s.predict_proba(X_test) / 3

    dart_s.fit(Xf_tr, yf_tr)
    oof_dart[val_idx] = dart_s.predict_proba(Xf_val)
    tst_dart          += dart_s.predict_proba(X_test) / 3

# Meta-learner: LR trained on OOF probability vectors
meta_train = np.hstack([oof_hgb, oof_dart])
meta_test  = np.hstack([tst_hgb, tst_dart])

lr_meta = LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced', random_state=42)
lr_meta.fit(meta_train, y)

stack_oof_preds = lr_meta.predict(meta_train)
stack_acc = accuracy_score(y, stack_oof_preds)
stack_f1  = f1_score(y, stack_oof_preds, average='macro')
print(f'\nStacking OOF eval — Acc={stack_acc:.4f}  F1={stack_f1:.4f}')
print(classification_report(y, stack_oof_preds, target_names=['Low', 'Medium', 'High']))

stack_preds = lr_meta.predict(meta_test)
stack_probs = lr_meta.predict_proba(meta_test)
print('Stacking submission:')
make_submission(test[ID_COL], stack_preds, 'submission_stacking_hw2.csv')

In [ ]:
print('Stacking summary: LR meta-learner on 3-fold OOF probs from HGB + DART')
print(f'  OOF Acc={stack_acc:.4f}  F1={stack_f1:.4f}')

---
## Probability Averaging Ensemble

In [ ]:
print('=' * 60)
print('PROBABILITY AVERAGING')
print('=' * 60)

# Equal-weight: HGB 33% + DART 33% + MLP 33%
avg_equal   = (hgb_probs + dart_probs + mlp_probs) / 3
print('\nEqual-weight (33/33/33):')
make_submission(test[ID_COL], avg_equal.argmax(axis=1), 'submission_avg_equal_hw2.csv')

# Custom-weight: tree models upweighted (typically better on tabular)
avg_weighted = 0.40*hgb_probs + 0.40*dart_probs + 0.20*mlp_probs
print('\nCustom-weight (HGB 40% + DART 40% + MLP 20%):')
make_submission(test[ID_COL], avg_weighted.argmax(axis=1), 'submission_avg_weighted_hw2.csv')

# Quick holdout validation using the same HGB holdout split
hgb_val_probs  = hgb_best_eval.predict_proba(Xv_h)
dart_val_probs = dart_best.predict_proba(Xv_h)
mlp_val_probs  = mlp_tuned.predict_proba(Xv_h)

eq_val = (hgb_val_probs + dart_val_probs + mlp_val_probs) / 3
wt_val = 0.4*hgb_val_probs + 0.4*dart_val_probs + 0.2*mlp_val_probs

print(f'\nHoldout validation (same 20% split as HGB):')
print(f'  Equal-weight    — Acc={accuracy_score(yv_h, eq_val.argmax(1)):.4f}  '
      f'F1={f1_score(yv_h, eq_val.argmax(1), average="macro"):.4f}')
print(f'  Custom-weight   — Acc={accuracy_score(yv_h, wt_val.argmax(1)):.4f}  '
      f'F1={f1_score(yv_h, wt_val.argmax(1), average="macro"):.4f}')

---
## Full Results Summary

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Random Forest (Phase 1)',
        'LightGBM gbdt (Phase 1)',
        'XGBoost (Phase 2)',
        'CatBoost (Phase 2)',
        'MLP Neural Net (Baseline)',
        'MLP Neural Net (Tuned)',
        'HistGradientBoosting (Baseline)',
        'HistGradientBoosting (Tuned)',
        'LightGBM DART (Baseline)',
        'LightGBM DART (Tuned)',
        'Stacking (HGB + DART + MLP)',
        'Avg Ensemble — Equal Weight',
        'Avg Ensemble — Custom Weight',
    ],
    'Phase': [1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2],
    'CV / Holdout Acc': [
        0.9855, 0.9847, 0.9846, 0.9839,
        mlp_base_acc, mlp_acc,
        hgb_base_acc, hgb_acc,
        dart_base_acc, dart_acc,
        stack_acc,
        None, None  # fill in after submission
    ],
    'CV / Holdout F1': [
        0.9710, 0.9701, 0.9700, 0.9686,
        mlp_base_f1, mlp_f1,
        hgb_base_f1, hgb_f1,
        dart_base_f1, dart_f1,
        stack_f1,
        None, None
    ],
    'Kaggle LB': [
        0.95876, 0.95937, 0.95875, 0.95615,
        None, 'TBD',
        None, 'TBD',
        None, 'TBD',
        'TBD', 'TBD', 'TBD'
    ]
})

print(results.to_string(index=False))

In [ ]:
# Visualise HW2 model comparison
hw2 = results[results['Phase'] == 2].copy()
hw2_plot = hw2.dropna(subset=['CV / Holdout F1']).reset_index(drop=True)

is_ensemble = hw2_plot['Model'].str.contains('Ensemble|Stacking|Avg')
colors_acc = ['#2196F3' if not e else '#FF9800' for e in is_ensemble]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

ax1.barh(hw2_plot['Model'], hw2_plot['CV / Holdout Acc'], color=colors_acc, alpha=0.85)
ax1.axvline(0.9847, color='red', linestyle='--', lw=1.5, alpha=0.8, label='LightGBM Phase 1 best (0.9847)')
ax1.set_xlabel('Accuracy', fontsize=11)
ax1.set_title('HW2 Model Accuracy\n(blue=individual, orange=ensemble)', fontsize=12)
ax1.legend(fontsize=9)
ax1.set_xlim([0.96, 1.0])
ax1.grid(axis='x', alpha=0.3)

ax2.barh(hw2_plot['Model'], hw2_plot['CV / Holdout F1'], color=colors_acc, alpha=0.85)
ax2.axvline(0.9701, color='red', linestyle='--', lw=1.5, alpha=0.8, label='LightGBM Phase 1 best (0.9701)')
ax2.set_xlabel('F1 Macro', fontsize=11)
ax2.set_title('HW2 Model F1 Macro\n(blue=individual, orange=ensemble)', fontsize=12)
ax2.legend(fontsize=9)
ax2.set_xlim([0.95, 1.0])
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../Homework 2/hw2_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to ../Homework 2/hw2_model_comparison.png')

---
## Discussion & Conclusions

### Feature Engineering
- **ET proxy** (evapotranspiration) and **water balance** were among the highest SHAP contributors — confirming the domain hypothesis that irrigation need is driven by evaporative demand vs. water supply
- **Log transforms** of `Rainfall_mm`, `Wind_Speed_kmh`, and `Previous_Irrigation_mm` improved tree splits on skewed distributions
- **Target encoding** of `Crop_Type`, `Season`, and their cross-feature `crop_stage` added meaningful ordinal signal
- **Leakage investigation:** `Irrigation_Type` and `Water_Source` showed [small/large] SHAP importance — decision: [include/exclude] in final models

### Model Performance
| Model | Key strength | Key weakness |
|---|---|---|
| MLP | Smooth boundaries; handles dense interactions | Needs scaling; slower; less interpretable |
| HistGBM | Fast; native class weighting; deterministic | Less tunable than LGB; no monotonic feature support here |
| DART | Strong regularization via dropout; diverse trees | Slower than gbdt; `drop_rate` sensitive |

### Ensemble
- **Stacking** uses a Logistic Regression meta-learner to learn the optimal combination of base model probabilities
- **Custom-weight averaging** (40/40/20 for HGB/DART/MLP) reflects tree models' typical advantage on tabular data
- Compare holdout scores: stacking vs. individual models to assess whether the combination adds meaningful value

### Moving Forward
- The best single model (update after submissions) will anchor future work
- Feature engineering gains (~[X] F1 points over base features) show domain-inspired features matter for this dataset
- DART regularization is worth exploring further: `uniform_drop=True` and higher `drop_rate` may help the minority 'High' class